# Evidential 3D Detector — Pretrained PointPillars (Model 2)

Colab notebook for running the **pretrained-backbone** evidential detector.

**Run All** does everything automatically:
1. Mount Drive
2. Install dependencies
3. Extract project zip from Drive
4. Download pretrained PointPillars checkpoint (if missing)
5. Auto-resume training from the latest checkpoint (or start fresh)
6. Evaluate best_model.pth and save plots to Drive

Config version is stamped to auto-wipe stale checkpoints when config changes.

## Cell 1 — Mount Drive + install deps

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install -q pyyaml tqdm tensorboardX gdown scikit-learn

## Cell 2 — Extract project zip from Drive
Upload `evidential_3d_pretrained.zip` to your Drive at `MyDrive/evidential_3d_pretrained.zip` before running.

In [ ]:
import os, shutil, zipfile

ZIP_PATH = '/content/drive/MyDrive/evidential_3d_pretrained.zip'
DEST = '/content/evidential_3d_pretrained'

assert os.path.exists(ZIP_PATH), f'Upload {ZIP_PATH} first!'

# Always extract fresh so code updates propagate
if os.path.exists(DEST):
    shutil.rmtree(DEST)

with zipfile.ZipFile(ZIP_PATH, 'r') as z:
    z.extractall('/content/')

# Some zip tools wrap everything in an outer folder — handle both cases
if not os.path.exists(os.path.join(DEST, 'tools', 'train.py')):
    # Look for a nested folder
    for entry in os.listdir('/content/'):
        inner = os.path.join('/content/', entry)
        if os.path.isdir(inner) and os.path.exists(os.path.join(inner, 'tools', 'train.py')):
            if inner != DEST:
                if os.path.exists(DEST):
                    shutil.rmtree(DEST)
                shutil.move(inner, DEST)
            break

assert os.path.exists(os.path.join(DEST, 'tools', 'train.py')), 'Extraction failed'
print('Project extracted to', DEST)
!ls {DEST}

## Cell 3 — Wire up Drive paths (data, checkpoints, outputs)

In [ ]:
import os, yaml

PROJECT = '/content/evidential_3d_pretrained'
DRIVE_ROOT = '/content/drive/MyDrive/evidential_3d_pretrained'
os.makedirs(DRIVE_ROOT, exist_ok=True)

# Persistent folders on Drive
DRIVE_CKPT = f'{DRIVE_ROOT}/checkpoints'
DRIVE_LOGS = f'{DRIVE_ROOT}/logs'
DRIVE_EVAL = f'{DRIVE_ROOT}/eval'
DRIVE_VIZ  = f'{DRIVE_ROOT}/viz'
DRIVE_PRETRAINED = f'{DRIVE_ROOT}/pretrained'
for d in [DRIVE_CKPT, DRIVE_LOGS, DRIVE_EVAL, DRIVE_VIZ, DRIVE_PRETRAINED]:
    os.makedirs(d, exist_ok=True)

# KITTI data — reuse the one already in Drive from Model 1 if present
KITTI_DRIVE_CANDIDATES = [
    '/content/drive/MyDrive/uncertainty_3d_detection/data/kitti',
    '/content/drive/MyDrive/evidential_3d_pretrained/data/kitti',
    '/content/drive/MyDrive/kitti',
]
KITTI_PATH = None
for c in KITTI_DRIVE_CANDIDATES:
    if os.path.exists(os.path.join(c, 'training', 'velodyne')):
        KITTI_PATH = c
        break
assert KITTI_PATH is not None, (
    'KITTI dataset not found on Drive. Expected at one of: ' + str(KITTI_DRIVE_CANDIDATES))
print('KITTI data:', KITTI_PATH)

# Symlink data into project
os.makedirs(f'{PROJECT}/data', exist_ok=True)
if not os.path.exists(f'{PROJECT}/data/kitti'):
    os.symlink(KITTI_PATH, f'{PROJECT}/data/kitti')

# Write a Colab-specific config with paths pointing to Drive
with open(f'{PROJECT}/configs/pretrained_kitti.yaml', 'r') as f:
    cfg = yaml.safe_load(f)

cfg['paths']['output_dir'] = DRIVE_ROOT
cfg['paths']['log_dir'] = DRIVE_LOGS
cfg['paths']['checkpoint_dir'] = DRIVE_CKPT
cfg['paths']['eval_dir'] = DRIVE_EVAL
cfg['paths']['viz_dir'] = DRIVE_VIZ
cfg['pretrained']['checkpoint_path'] = f'{DRIVE_PRETRAINED}/pointpillar_7728.pth'
cfg['data']['data_path'] = f'{PROJECT}/data/kitti'

COLAB_CONFIG = '/content/colab_config.yaml'
with open(COLAB_CONFIG, 'w') as f:
    yaml.safe_dump(cfg, f, sort_keys=False)

print('Config written to', COLAB_CONFIG)
print('Checkpoints →', DRIVE_CKPT)
print('Pretrained  →', DRIVE_PRETRAINED)

## Cell 4 — Config versioning + auto-resume decision

If the config version changed (e.g. you tuned LR), wipe stale checkpoints.
Otherwise, detect whether to resume or start fresh.

In [ ]:
import os, re, glob

# Bump this whenever you change training-affecting config (LR, loss weights, epochs)
CONFIG_VERSION = 'v2_pretrained_stage1-20_stage2-0_lr1e-3_frozen_only'

marker_file = os.path.join(DRIVE_CKPT, '.config_version')

current_version = None
if os.path.exists(marker_file):
    with open(marker_file) as f:
        current_version = f.read().strip()

if current_version is None:
    print('No config marker yet — assuming fresh start.')
    with open(marker_file, 'w') as f:
        f.write(CONFIG_VERSION)
elif current_version != CONFIG_VERSION:
    print(f'Config changed ({current_version} → {CONFIG_VERSION})')
    print('Wiping stale checkpoints...')
    for p in glob.glob(os.path.join(DRIVE_CKPT, '*.pth')):
        os.remove(p)
        print(f'  removed {os.path.basename(p)}')
    with open(marker_file, 'w') as f:
        f.write(CONFIG_VERSION)
else:
    print(f'Config version matches: {CONFIG_VERSION}')

# Scan existing checkpoints — NUMERIC sort (avoids '9' > '12' bug)
def _epoch_num(p):
    m = re.search(r'checkpoint_epoch_(\d+)\.pth', os.path.basename(p))
    return int(m.group(1)) if m else -1

epoch_ckpts = sorted(
    glob.glob(os.path.join(DRIVE_CKPT, 'checkpoint_epoch_*.pth')),
    key=_epoch_num,
)
latest_ckpt = os.path.join(DRIVE_CKPT, 'latest.pth')

if epoch_ckpts or os.path.exists(latest_ckpt):
    print('\n=== SESSION RESUME ===')
    if epoch_ckpts:
        print(f'Last completed: {os.path.basename(epoch_ckpts[-1])}')
    if os.path.exists(latest_ckpt):
        print('latest.pth also present (mid-epoch save)')
    print('train.py will auto-resume from the newest.')
else:
    print('\n=== FRESH START ===')
    print('No checkpoints found — training will start from pretrained backbone.')

## Cell 5 — Download pretrained PointPillars checkpoint
Only runs if the file isn't already on Drive.

In [ ]:
import sys
sys.path.insert(0, PROJECT)

from models.pretrained_loader import download_pretrained
pretrained_path = f'{DRIVE_PRETRAINED}/pointpillar_7728.pth'
download_pretrained(pretrained_path)
print('Pretrained ready at', pretrained_path)
print('Size:', os.path.getsize(pretrained_path) / 1e6, 'MB')

## Cell 6 — Train (auto-resumes if checkpoints exist)

In [ ]:
%cd {PROJECT}
!python tools/train.py --config {COLAB_CONFIG}

## Cell 7 — Evaluate best_model.pth

In [ ]:
BEST = f'{DRIVE_CKPT}/best_model.pth'
assert os.path.exists(BEST), f'No best_model.pth at {BEST}'

%cd {PROJECT}
!python tools/evaluate.py \
    --config {COLAB_CONFIG} \
    --checkpoint {BEST} \
    --visualize --num_viz 15